# Multilingual MiniLM — with Hyperparameter Tuning

Optuna search over learning rate, batch size, dropout, warmup ratio and weight decay, maximising
**validation macro-F1**. The best configuration found here is then used for a full-length final
training run. This is the HPO companion to `Multilingual_MiniLM.ipynb` and mirrors the
`*_hyperparameters.ipynb` notebooks of the other models.

> **Model-size note.** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` is
> multilingually pre-trained on 50+ languages **including Filipino/Tagalog**, so it is a
> language-fair peer of the Tagalog-native models. Its trade-off is **size**: a ~250k-token
> multilingual vocabulary makes it the **largest base in the study (~118M params)**. Treat it as
> the multilingual-transformer benchmark on the accuracy-vs-size curve. The HPO protocol
> (70/15/15 split, macro metrics, validation-calibrated threshold, shared litert-torch export)
> is identical to the other models so results stay directly comparable.

In [ ]:
!pip install onnxscript optimum[onnxruntime] onnxconverter-common litert-torch torchao optuna

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import time
warnings.filterwarnings('ignore')

In [ ]:
try:
    from optimum.onnxruntime import ORTModelForSequenceClassification
    from optimum.onnxruntime.configuration import OptimizationConfig
    ONNX_AVAILABLE = True
except ImportError:
    print("Optimum ONNX Runtime not available, ONNX export will be limited")
    ONNX_AVAILABLE = False

In [ ]:
# Set random seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# Multilingual MiniLM Classifier Model
# Base: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
#   - 12 transformer layers, hidden size 384, ~250k multilingual WordPiece/SP vocab
#   - multilingually pre-trained (50+ languages incl. Filipino/Tagalog)
# A custom [CLS]-pooling head is used (same head design as the TinyBERT baseline),
# so the architecture stays MiniLM's own while the training/eval/export wrapper
# matches the rest of the study.
class MiniLMClassifier(nn.Module):
    def __init__(self, bert_model_name, num_labels=2, dropout=0.3):
        super(MiniLMClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # Use [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
# Custom Dataset Class
class NSFWDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Training Function
def train_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    predictions, true_labels = [], []

    progress_bar = tqdm(dataloader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)

    return avg_loss, accuracy

In [ ]:
# Evaluation Function
# Returns class probabilities as well, so the validation set can be used to
# calibrate a decision threshold (matching the RoBERTa / DistilBERT / LSTM / BiLSTM protocol).
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    predictions, true_labels, probabilities = [], [], []

    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="Evaluating")
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
            probabilities.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(dataloader)

    return avg_loss, predictions, true_labels, np.array(probabilities)

In [ ]:
# Validation-set threshold calibration (mirrors RoBERTa / DistilBERT / LSTM / BiLSTM)
def find_optimal_threshold(probs, y_true):
    """Pick the probability threshold that maximises macro-F1 on the validation set.

    Searches a sane band and only deviates from the neutral 0.5 if the validation gain
    exceeds 0.01, so the operating point matches the policy used by the other models
    and the test numbers stay directly comparable.
    """
    y_true = np.asarray(y_true)
    pos = np.asarray(probs)[:, 1]

    thresholds = np.arange(0.30, 0.71, 0.01)
    f1s = [f1_score(y_true, (pos >= t).astype(int), average='macro', zero_division=0) for t in thresholds]
    best_idx = int(np.argmax(f1s))
    tuned_t, tuned_f1 = float(thresholds[best_idx]), f1s[best_idx]

    f1_half = f1_score(y_true, (pos >= 0.5).astype(int), average='macro', zero_division=0)
    if tuned_f1 - f1_half > 0.01:
        best_t, best_f1 = tuned_t, tuned_f1
    else:
        best_t, best_f1 = 0.5, f1_half

    print(f" Optimal Threshold: {best_t:.2f} (tuned {tuned_t:.2f}->F1 {tuned_f1:.4f} vs 0.50->F1 {f1_half:.4f})")
    print(f" Best Validation Macro-F1: {best_f1:.4f}")
    return best_t

In [ ]:
# Plot Confusion Matrix
def plot_confusion_matrix(cm, save_path='confusion_matrix_minilm.png'):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['SFW', 'NSFW'],
                yticklabels=['SFW', 'NSFW'],
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.title('Confusion Matrix - Multilingual MiniLM NSFW Classifier', fontsize=14, fontweight='bold')

    # Add percentage annotations
    total = cm.sum()
    for i in range(2):
        for j in range(2):
            percentage = (cm[i, j] / total) * 100
            plt.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)',
                    ha='center', va='center', fontsize=10, color='gray')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f" Confusion matrix saved to {save_path}")
    plt.close()

In [ ]:
# Plot Training History
def plot_training_history(train_losses, val_losses, train_accs, val_accs,
                         save_path='training_history_minilm.png'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    epochs = range(1, len(train_losses) + 1)

    # Loss plot
    ax1.plot(epochs, train_losses, 'b-o', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-o', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Accuracy plot
    ax2.plot(epochs, train_accs, 'b-o', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accs, 'r-o', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f" Training history saved to {save_path}")
    plt.close()

In [ ]:
# Inference Function
def predict(text, model, tokenizer, device, max_length=128):
    model.eval()

    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    start_time = time.time()
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1)
    inference_time = (time.time() - start_time) * 1000  # Convert to milliseconds

    return pred.item(), probs.cpu().numpy()[0], inference_time

In [ ]:
# Batch Inference Function
def batch_predict(texts, model, tokenizer, device, max_length=128, batch_size=32):
    model.eval()
    all_predictions = []
    all_probabilities = []

    dataset = NSFWDataset(texts, [0] * len(texts), tokenizer, max_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Batch Inference"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_predictions.extend(preds.cpu().numpy())
            all_probabilities.extend(probs.cpu().numpy())

    return all_predictions, all_probabilities

In [ ]:
# Export to ONNX Function
def export_to_onnx(model, tokenizer, onnx_path, device):
    import os
    print("Exporting Multilingual MiniLM NSFW Classifier to ONNX format...")

    try:
        # Create dummy input with sample Tagalog text (same max_length as training: 128)
        dummy_input = tokenizer(
            "Magandang umaga sa lahat",
            return_tensors="pt",
            max_length=128,
            padding="max_length",
            truncation=True
        )

        input_ids = dummy_input["input_ids"].to(device)
        attention_mask = dummy_input["attention_mask"].to(device)

        model.eval()

        os.makedirs(os.path.dirname(onnx_path), exist_ok=True)
        torch.onnx.export(
            model,
            (input_ids, attention_mask),
            onnx_path,
            export_params=True,
            opset_version=18,
            do_constant_folding=True,
            input_names=['input_ids', 'attention_mask'],
            output_names=['logits'],
            dynamic_axes={
                'input_ids': {0: 'batch_size', 1: 'sequence'},
                'attention_mask': {0: 'batch_size', 1: 'sequence'},
                'logits': {0: 'batch_size'}
            }
        )

        print(f"ONNX model exported to: {onnx_path}")

        # Report a FAIR size: include the external weight sidecar (.onnx.data) if
        # the exporter spilled the weights into one, otherwise the headline number
        # is misleadingly small (the graph alone is < 1 MB).
        total_bytes = os.path.getsize(onnx_path)
        sidecar = onnx_path + ".data"
        if os.path.exists(sidecar):
            total_bytes += os.path.getsize(sidecar)
            print(f"ONNX external weights: {sidecar}")
        print(f"ONNX model size (graph + weights): {total_bytes / (1024 * 1024):.2f} MB")

        return True

    except Exception as e:
        print(f"ONNX export failed: {e}")
        import traceback
        traceback.print_exc()
        return False

In [ ]:
# Export to TFLite via litert-torch (ai-edge-torch) -- same route as the other transformers
def export_to_tflite_ai_edge(model, model_name, output_dir, max_length=128):
    """Convert the trained MiniLM classifier to TFLite via litert-torch (LiteRT).

    transformers v5 dropped native TF, so the old TFAutoModel(from_pt=True) -> TFLite
    route no longer works (and previously produced a bloated ~224 MB FP16 file).
    litert-torch (formerly 'ai-edge-torch') converts PyTorch -> LiteRT directly and
    supports a data-free dynamic-range INT8 recipe. This matches the DistilBERT /
    DOST_RoBERTa / TinyBERT export path, so every transformer in the study ships
    comparable artifacts and we can report a fair on-device size.

    Emits:
      - minilm_nsfw_float32.tflite : reference build (largest)
      - minilm_nsfw_int8.tflite    : dynamic-range INT8 weights (~1/4 size)
    Both import via Android Studio's ML Model Binding.
    """
    import importlib, os
    print("Exporting Multilingual MiniLM NSFW Classifier to TFLite (litert-torch)...")

    # 'ai-edge-torch' was renamed to 'litert-torch'; the old name is now a stub
    # without .convert. Use whichever module actually provides the API.
    aet = None
    for modname in ('litert_torch', 'ai_edge_torch'):
        try:
            m = importlib.import_module(modname)
        except Exception:
            continue
        if hasattr(m, 'convert'):
            aet = m
            break
    if aet is None:
        print("No working litert_torch/ai_edge_torch (.convert) found; skipping TFLite export.")
        return False
    pkg = aet.__name__
    print(f"  Using '{pkg}' for TFLite conversion.")

    try:
        # Rebuild a clean CPU eval copy with eager attention. eager avoids SDPA's
        # data-dependent branches, which torch.export cannot trace. The custom
        # MiniLMClassifier.forward already returns the logits tensor directly,
        # so no logits-unwrapping wrapper is needed.
        export_model = MiniLMClassifier(model_name, num_labels=2)
        export_model.bert = AutoModel.from_pretrained(model_name, attn_implementation="eager")
        export_model.load_state_dict(model.state_dict())
        export_model.eval().to("cpu")

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Fixed [1, max_length] sample -> the .tflite takes a fixed-length input.
        enc = tokenizer(
            "Ito ay isang sample na Tagalog text", return_tensors="pt",
            max_length=max_length, padding="max_length", truncation=True
        )
        sample = (enc["input_ids"], enc["attention_mask"])

        os.makedirs(output_dir, exist_ok=True)
        ok = False

        # --- Float32 LiteRT model (reliable baseline) ---
        try:
            print(f"  Converting PyTorch -> TFLite (float32) via {pkg}...")
            edge = aet.convert(export_model, sample)
            f32 = os.path.join(output_dir, "minilm_nsfw_float32.tflite")
            edge.export(f32)
            print(f"  TFLite (float32) saved to {f32} ({os.path.getsize(f32) / (1024 * 1024):.2f} MB)")
            ok = True
        except Exception as e:
            print(f"  {pkg} float32 conversion failed: {e}")
            import traceback
            traceback.print_exc()

        # --- Dynamic-range INT8 (data-free; ~1/4 size) ---
        try:
            pt2e = importlib.import_module(f"{pkg}.quantize.pt2e_quantizer")
            qcfg = importlib.import_module(f"{pkg}.quantize.quant_config")
            PT2EQuantizer = pt2e.PT2EQuantizer
            get_symmetric_quantization_config = pt2e.get_symmetric_quantization_config
            QuantConfig = qcfg.QuantConfig

            # prepare_pt2e/convert_pt2e moved out of torch.ao into torchao in newer torch.
            prepare_pt2e = convert_pt2e = None
            for modpath in (
                'torchao.quantization.pt2e.quantize_pt2e',
                'torch.ao.quantization.quantize_pt2e',
                'torch.ao.quantization.pt2e.quantize_pt2e',
            ):
                try:
                    qmod = importlib.import_module(modpath)
                    prepare_pt2e, convert_pt2e = qmod.prepare_pt2e, qmod.convert_pt2e
                    break
                except Exception:
                    continue
            if prepare_pt2e is None:
                raise ImportError("prepare_pt2e/convert_pt2e not found (torch.ao or torchao)")

            print(f"  Converting PyTorch -> TFLite (dynamic-range INT8) via {pkg} PT2E...")
            quantizer = PT2EQuantizer().set_global(
                get_symmetric_quantization_config(is_per_channel=True, is_dynamic=True)
            )

            # Capture a pre-autograd graph (entry point name varies across torch versions).
            captured = None
            try:
                captured = torch.export.export_for_training(export_model, sample).module()
            except (AttributeError, TypeError):
                try:
                    from torch._export import capture_pre_autograd_graph
                    captured = capture_pre_autograd_graph(export_model, sample)
                except Exception:
                    captured = torch.export.export(export_model, sample).module()

            prepared = prepare_pt2e(captured, quantizer)
            converted = convert_pt2e(prepared)
            edge_q = aet.convert(
                converted, sample, quant_config=QuantConfig(pt2e_quantizer=quantizer)
            )
            i8 = os.path.join(output_dir, "minilm_nsfw_int8.tflite")
            edge_q.export(i8)
            print(f"  TFLite (INT8) saved to {i8} ({os.path.getsize(i8) / (1024 * 1024):.2f} MB)")
            ok = True
        except Exception as e:
            print(f"  {pkg} INT8 conversion failed: {e}")
            import traceback
            traceback.print_exc()

        if ok:
            print("  Import in Android Studio via: File > New > Other > TensorFlow Lite Model.")
        else:
            print(f"  {pkg} produced no TFLite files.")
        return ok

    except Exception as e:
        print(f"TFLite export failed: {e}")
        import traceback
        traceback.print_exc()
        return False

In [ ]:
# Optuna hyperparameter search over the MiniLM fine-tuning recipe
def run_hpo(X_train, y_train, X_val, y_val, model_name, device,
            n_trials=15, hpo_epochs=3, max_length=128):
    """Search lr / batch size / dropout / warmup ratio / weight decay, maximising
    validation macro-F1. Mirrors the Optuna setup used for the other models; the
    best params are then used for a full-length final training run. MiniLM-L12 is
    the largest base in the study, so batch sizes are kept modest for Colab GPUs.
    """
    import optuna, os

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def objective(trial):
        lr = trial.suggest_float('learning_rate', 1e-5, 5e-5, log=True)
        batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
        dropout = trial.suggest_float('dropout', 0.1, 0.5)
        warmup_ratio = trial.suggest_float('warmup_ratio', 0.0, 0.2)
        weight_decay = trial.suggest_float('weight_decay', 0.0, 0.3)

        set_seed(42)
        model = MiniLMClassifier(model_name, num_labels=2, dropout=dropout).to(device)

        train_loader = DataLoader(
            NSFWDataset(X_train, y_train, tokenizer, max_length),
            batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(
            NSFWDataset(X_val, y_val, tokenizer, max_length),
            batch_size=batch_size)

        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        total_steps = len(train_loader) * hpo_epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer, int(warmup_ratio * total_steps), total_steps)
        criterion = nn.CrossEntropyLoss()

        best_f1 = 0.0
        for epoch in range(hpo_epochs):
            train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
            _, val_preds, val_labels, _ = evaluate(model, val_loader, criterion, device)
            f1m = f1_score(val_labels, val_preds, average='macro', zero_division=0)
            best_f1 = max(best_f1, f1m)
            trial.report(f1m, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        # Free GPU memory between trials (MiniLM-L12 is large)
        try:
            del model, train_loader, val_loader
            import gc
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

        return best_f1

    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))
    study.optimize(objective, n_trials=n_trials)

    os.makedirs('tuning_results', exist_ok=True)
    try:
        study.trials_dataframe().to_csv('tuning_results/hpo_trials_minilm.csv', index=False)
        print("HPO trials saved to tuning_results/hpo_trials_minilm.csv")
    except Exception as e:
        print(f"Could not save HPO trials: {e}")

    print(f"\nBest trial #{study.best_trial.number} | Validation Macro-F1: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")
    return study.best_params

In [ ]:
# Main pipeline: HPO -> full training -> calibrated test evaluation -> export
def main(csv_path, final_epochs=10, n_trials=15, max_length=128):
    import os, sys, json, time

    if not os.path.exists(csv_path):
        sys.exit(f"Dataset not found: {csv_path}")

    print(" Loading dataset.")
    df = pd.read_csv(csv_path)
    if df.iloc[:, 1].dtype == object:
        label_map = {'safe': 0, 'nsfw': 1}
        df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.lower().str.strip().map(label_map)
        df = df.dropna(subset=[df.columns[1]])
        df.iloc[:, 1] = df.iloc[:, 1].astype(int)
    texts = df.iloc[:, 0].values
    labels = df.iloc[:, 1].values
    print(f"  Total samples: {len(texts):,} | NSFW: {(labels==1).sum():,} | SFW: {(labels==0).sum():,}")

    # 70/15/15 stratified -- identical protocol to the other six models
    X_train, X_temp, y_train, y_temp = train_test_split(
        texts, labels, test_size=0.3, random_state=42, stratify=labels)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    print(f"  Train {len(X_train):,} | Val {len(X_val):,} | Test {len(X_test):,}")

    model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("\n  NOTE: 'paraphrase-multilingual-MiniLM-L12-v2' is multilingually pre-trained on 50+")
    print("        languages INCLUDING Filipino/Tagalog -- a language-fair peer of the Tagalog-native")
    print("        models. Its trade-off is SIZE: ~118M params, the largest base in the study.")

    # ---------------- Hyperparameter Optimization ----------------
    print(f"\n Running Optuna HPO ({n_trials} trials)...")
    best_params = run_hpo(X_train, y_train, X_val, y_val, model_name, device,
                          n_trials=n_trials, hpo_epochs=3, max_length=max_length)

    os.makedirs('models', exist_ok=True)
    with open('models/minilm_hpo_results.json', 'w') as f:
        json.dump(best_params, f, indent=2)
    print(" Saved best hyperparameters to models/minilm_hpo_results.json")

    lr = best_params['learning_rate']
    batch_size = best_params['batch_size']
    dropout = best_params['dropout']
    warmup_ratio = best_params['warmup_ratio']
    weight_decay = best_params['weight_decay']

    # ---------------- Final training with best params ----------------
    print("\n Final training with best hyperparameters...")
    set_seed(42)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = MiniLMClassifier(model_name, num_labels=2, dropout=dropout).to(device)

    train_loader = DataLoader(NSFWDataset(X_train, y_train, tokenizer, max_length), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(NSFWDataset(X_val, y_val, tokenizer, max_length), batch_size=batch_size)
    test_loader = DataLoader(NSFWDataset(X_test, y_test, tokenizer, max_length), batch_size=batch_size)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = len(train_loader) * final_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(warmup_ratio * total_steps), total_steps)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    for epoch in range(final_epochs):
        print(f"\n EPOCH {epoch+1}/{final_epochs}")
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        val_loss, val_preds, val_labels, _ = evaluate(model, val_loader, criterion, device)
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
        train_losses.append(train_loss); val_losses.append(val_loss)
        train_accs.append(train_acc); val_accs.append(val_acc)
        print(f"  Train Acc {train_acc*100:.2f}% | Val Acc {val_acc*100:.2f}% | Val Macro-F1 {val_f1:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({'model_state_dict': model.state_dict(), 'best_params': best_params},
                       'best_minilm_tuned_model.pth')
            tokenizer.save_pretrained('best_minilm_tuned_tokenizer')
            print("   New best model saved.")

    plot_training_history(train_losses, val_losses, train_accs, val_accs,
                          save_path='training_history_minilm_tuned.png')

    # Reload best + calibrate threshold on the validation set
    model.load_state_dict(torch.load('best_minilm_tuned_model.pth')['model_state_dict'])
    print("\n Calibrating decision threshold on the validation set.")
    _, _, val_labels_full, val_probs = evaluate(model, val_loader, criterion, device)
    best_threshold = find_optimal_threshold(val_probs, val_labels_full)

    # Final test evaluation with calibrated threshold + macro metrics
    print("\n Final test evaluation.")
    start = time.time()
    _, _, test_labels, test_probs = evaluate(model, test_loader, criterion, device)
    runtime = time.time() - start
    test_preds = (test_probs[:, 1] >= best_threshold).astype(int)

    accuracy = accuracy_score(test_labels, test_preds)
    f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    f1_weighted = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
    precision_macro = precision_score(test_labels, test_preds, average='macro', zero_division=0)
    recall_macro = recall_score(test_labels, test_preds, average='macro', zero_division=0)
    cm = confusion_matrix(test_labels, test_preds)

    print("FINAL EVALUATION (TEST SET) -- tuned Multilingual MiniLM")
    print(f"  Decision threshold (validation-calibrated): {best_threshold:.2f}")
    print(f"  Accuracy:            {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision (Macro):   {precision_macro:.4f}")
    print(f"  Recall (Macro):      {recall_macro:.4f}")
    print(f"  F1-Score (Macro):    {f1_macro:.4f}")
    print(f"  F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"  Latency/sample: {runtime/len(test_labels)*1000:.4f} ms | {len(test_labels)/runtime:.1f} samples/s")
    print("\n" + classification_report(test_labels, test_preds, target_names=['SFW', 'NSFW'], digits=4))
    plot_confusion_matrix(cm, save_path='confusion_matrix_minilm_tuned.png')

    # Deployment config (threshold + base) mirrors the other transformers
    with open('minilm_tuned_deployment_config.json', 'w') as f:
        json.dump({'decision_threshold': float(best_threshold),
                   'max_length': max_length, 'base_model': model_name,
                   'num_labels': 2, 'best_params': best_params,
                   'language_note': 'Multilingual MiniLM base (50+ langs incl. Filipino/Tagalog); largest base in study'},
                  f, indent=2)

    # Export (shared routes)
    print("\n EXPORTING MODELS")
    export_to_onnx(model, tokenizer, "exported_models/minilm_nsfw_tuned.onnx", device)
    export_to_tflite_ai_edge(model, model_name, "exported_models", max_length=max_length)

    print("\n DONE.")
    print(f"  Best params: {best_params}")
    print(f"  Test accuracy: {accuracy:.4f} | Macro-F1: {f1_macro:.4f} | threshold {best_threshold:.2f}")
    return model, tokenizer, best_params

In [ ]:
if __name__ == "__main__":
    CSV_PATH = "cleaned_data.csv"

    model, tokenizer, best_params = main(
        csv_path=CSV_PATH,
        final_epochs=10,   # full-length training with the tuned config
        n_trials=15,       # Optuna trials (each trains a few short epochs)
        max_length=128,
    )

In [ ]:
# Zip the results folder
!zip -r Multilingual_MiniLM_hyperparameters.zip /content/

In [ ]:
from google.colab import files

files.download('Multilingual_MiniLM_hyperparameters.zip')